In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from neo4j import GraphDatabase, Result
from tqdm import tqdm
from typing import Dict, Any
from langchain_community.graphs import Neo4jGraph
from langchain_community.vectorstores import Neo4jVector

import pandas as pd

In [ ]:
# user defined imports
from promptsd.causal_literature_prompts import *
import helpers

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

## Parameters

In [ ]:
path = "~/kg_aug_causal_disc_exp"

## Setting Up Graph

In [ ]:
from config import NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD,NEO4J_DATABASE, DIRECTORY

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD),database=NEO4J_DATABASE)

def db_query(cypher: str, params: Dict[str, Any] = {}) -> pd.DataFrame:
    """Executes a Cypher statement and returns a DataFrame"""
    return driver.execute_query(
        cypher, parameters_=params, result_transformer_=Result.to_df
    )

In [ ]:
graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
    refresh_schema=False,
    driver_config={"notifications_disabled_classifications": ["DEPRECATION"]}
)

## Setting up LLM

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Literal

class Reasoning_Step(BaseModel):
    reasoning_step: str = Field(..., description="An intermediate reasoning step for breaking down the given context and query")

class Answer(BaseModel):
    reasoning: List[Reasoning_Step] = Field(..., description="List of reasoning steps")
    conclusion: Literal['A', 'B', 'C']

In [ ]:
from vllm.sampling_params import SamplingParams
from outlines import generate, samplers
from llm_client import get_client

generator = get_client(schema=Answer)
summarizer = get_client()

# Local Retriever

In [ ]:
# parameters for the local search query
from config import topChunks, topCommunities, topRels, topEntities

context = {}
lc_retrieval_query = helpers.load_query("local_search.cypher")
kw_retrieval_query = helpers.load_query("keyword_search.cypher")
path_retrieval_query = helpers.load_query("path_search.cypher")

In [ ]:
# variables of interest
with open("variable_definitions/default_definitions.json", "r") as file:
    def_map = json.load(file)

In [ ]:
db_query(
    """
    CREATE VECTOR INDEX vector IF NOT EXISTS
    FOR (n:__Entity__)
    ON n.embedding
    OPTIONS {indexConfig: {
      `vector.dimensions`: 768,
      `vector.similarity_function`: "cosine"
    }};
    """
)

In [ ]:
db_query("SHOW INDEXES")

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
import torch

embedding = HuggingFaceEmbeddings(
    model_name="pritamdeka/S-PubMedBert-MS-MARCO",
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

In [ ]:
lc_retrieval_query = helpers.load_query("local_search.cypher")
lc_vector = Neo4jVector.from_existing_index(
    embedding=embedding,
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
    index_name="vector", # may need to alter
    search_type="vector",
    # keyword_index_name="keyword",
    retrieval_query=lc_retrieval_query,
)

In [ ]:
from helpers import *
from build_context import stringify_report, format_triplet, construct_query_context
from config import query_context_window

# remember to clear the context cache when changing the summary prompt
context = {} # (var1, var2) -> summary

def retrieve_context(var1, var2, use_paths=False, debug=False) -> str:

    if (var1, var2) in context:
        return context[(var1, var2)]

    var1res = lc_vector.similarity_search(
        f"{def_map.get(var1, var1)}",
        k=topEntities,
        params={
            "topChunks": topChunks,
            "topCommunities": topCommunities,
            "topRels": topRels
        }
    )

    metadata = var1res[0].metadata
    reports = [stringify_report(report) for report in metadata["Reports"]]
    chunks = metadata["Chunks"]
    relationships = [format_triplet(triplet) for triplet in metadata["Relationships"]]
    context1 = construct_query_context(relationships, chunks, reports, max_context_window=query_context_window)

    var2res = lc_vector.similarity_search(
        f"{def_map.get(var2,var2)}",
        k=topEntities,
        params={
            "topChunks": topChunks,
            "topCommunities": topCommunities,
            "topRels": topRels
        }
    )

    metadata = var2res[0].metadata
    reports = [stringify_report(report) for report in metadata["Reports"]]
    chunks = metadata["Chunks"]
    relationships = [format_triplet(triplet) for triplet in metadata["Relationships"]]
    context2 = construct_query_context(relationships, chunks, reports, max_context_window=query_context_window)
    
    final_report = f"# Report for Variable 1: {var1}\n" + context1 + f"\n# Report for Variable 2: {var2}\n" + context2

    if debug:
        print(final_report)
    
    context[(var1, var2)] = final_report
    return final_report

In [ ]:
def local_retriever(query, var1, var2, summary, debug=False):
    if debug:
        print(reduce_causal_lit(query, var1, var2, summary, def_map))
        print(helpers.token_count(reduce_causal_lit(query, var1, var2, summary, def_map)))
    response = generator(reduce_causal_lit(query, var1, var2, summary, def_map), sampling_params={"n":1, "temperature":0.0, "top_k":1})

    return response.conclusion, helpers.reasoning_to_string_multiple_choice(response)

In [ ]:
def llm_retriever(query, var1, var2, debug=False):
    if debug:
        print(predict_causal_lit(query, var1, var2, def_map))
    response = generator(predict_causal_lit(query, var1, var2, def_map), sampling_params={"n":1, "temperature":0.0, "top_k":1})
    return response.conclusion, helpers.reasoning_to_string_multiple_choice(response)

In [ ]:
from promptsd.query_prompts import plausibility_prompt, temporality_prompt, causal_lit_prompt, association_prompt

def query_local_causality(row):
    var1, var2, label = row['var1'], row['var2'], row["label"]
    # bandaid for now
    var1 = "Sleep disturbance" if var1 == "Sleep" else var1
    var2 = "Sleep disturbance" if var2 == "Sleep" else var2

    report = retrieve_context(var1, var2)
    clquery = causal_lit_prompt(var1, var2)
    causal_literature, causal_lit_reasoning = local_retriever(clquery, var1, var2, report)
    return [var1, var2, causal_literature, causal_lit_reasoning, report, label]

In [ ]:
def query_llm_causality(row):
    var1, var2, label = row['var1'], row['var2'], row["label"]
    # bandaid for now
    var1 = "Sleep disturbance" if var1 == "Sleep" else var1
    var2 = "Sleep disturbance" if var2 == "Sleep" else var2
    
    clquery = causal_lit_prompt(var1, var2)
    causal_literature, causal_lit_reasoning = llm_retriever(clquery, var1, var2)
    return [var1, var2, causal_literature, causal_lit_reasoning, label]

In [ ]:
proto = pd.read_csv(f"{path}/data/proto_cleaned.csv").drop(columns=["Unnamed: 0"])
full = pd.read_csv(f"{path}/data/full_cleaned.csv").drop(columns=["Unnamed: 0"])

# Experiments

## LLM

In [ ]:
from prompts import *
res = full.apply(query_llm_causality, axis=1)

In [ ]:
columns = "Var1", "Var2", "Causal Literature",  "Causal Literature Reasoning", "Label"
llm_res = pd.DataFrame(res.to_list(), columns=columns)
llm_res.to_csv("results/llm_full_causal_literature.csv")
llm_res

## RAG

## Local Search

In [ ]:
from prompts import *
res = full.apply(query_local_causality, axis=1)

In [ ]:
columns = "Var1", "Var2", "Causal Literature",  "Causal Literature Reasoning", "Report", "Label"
local_res = pd.DataFrame(res.to_list(), columns=columns)
local_res.to_csv("results/kg+rag_full_causal_literature.csv")
local_res

# Saving the Prompts

In [ ]:
import json

# variables of interest
with open("variable_definitions/default_definitions.json", "r") as file:
    def_map = json.load(file)

In [ ]:
from promptsd.query_prompts import causal_lit_prompt
var1 = "Sex"
var2 = "Anxiety"

clquery = causal_lit_prompt(var1, var2)

In [ ]:
from prompts import *
from promptsd.causal_literature_prompts import reduce_causal_lit, predict_causal_lit, reduce_rag_causal_lit

with open("raw_prompts/kg+llm_causal_literature_prompt.txt", "w") as f:
    print(reduce_causal_lit(clquery, var1, var2, "", def_map), file=f)

with open("raw_prompts/llm+rag_causal_literature_prompt.txt", "w") as f:
    print(reduce_rag_causal_lit(clquery, var1, var2, "", def_map), file=f)

with open("raw_prompts/llm_prompt_causal_literature.txt", "w") as f:
    print(predict_causal_lit(clquery, var1, var2, def_map), file=f)